# Exploratory Data Analysis (EDA)
## Phishing Detection Dataset

**Student:** [Your Name]  
**Date:** February 2026  
**Lab:** AI in Cybersecurity Lab 2

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

# Add parent directory to path
sys.path.append('..')

# Configure display settings
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Enable inline plotting
%matplotlib inline

print("✓ Libraries imported successfully")

## 1. Load Dataset

In [ ]:
# Load the phishing dataset
df = pd.read_csv('../data/raw/phishing_dataset.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print(f"\nRows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\n" + "="*60)
print("First 5 rows:")
df.head()

## 2. Dataset Overview

In [ ]:
# Display dataset information
print("Dataset Information:")
print("="*60)
df.info()

print("\n\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# Display statistical summary
print("Statistical Summary:")
print("="*60)
df.describe()

## 3. Data Quality Check

In [ ]:
# Check for missing values
print("Missing Values per Column:")
print("="*60)
missing = df.isnull().sum()
print(missing)

print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Percentage missing: {(df.isnull().sum().sum() / df.size * 100):.2f}%")

if df.isnull().sum().sum() == 0:
    print("\n✓ No missing values found - Dataset is complete!")

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates == 0:
    print("✓ No duplicates found - Dataset is clean!")
else:
    print(f"⚠ Found {duplicates} duplicate rows")

## 4. Target Variable Analysis

In [ ]:
# Analyze target variable distribution
print("Class Distribution:")
print("="*60)
class_counts = df['is_phishing'].value_counts()
print(class_counts)

print("\nProportions:")
class_props = df['is_phishing'].value_counts(normalize=True)
print(class_props)

print(f"\nLegitimate URLs: {class_counts[0]} ({class_props[0]*100:.2f}%)")
print(f"Phishing URLs: {class_counts[1]} ({class_props[1]*100:.2f}%)")

# Calculate balance ratio
balance_ratio = min(class_counts) / max(class_counts)
print(f"\nClass Balance Ratio: {balance_ratio:.3f}")
if balance_ratio > 0.8:
    print("✓ Classes are well balanced")
elif balance_ratio > 0.5:
    print("⚠ Slight class imbalance")
else:
    print("⚠ Significant class imbalance - consider using SMOTE or class weights")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
class_counts.plot(kind='bar', ax=axes[0], color=['green', 'red'], alpha=0.7)
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class (0=Legitimate, 1=Phishing)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['Legitimate', 'Phishing'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
class_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                  colors=['green', 'red'], labels=['Legitimate', 'Phishing'],
                  startangle=90)
axes[1].set_title('Class Distribution (Proportion)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Plot saved to results/class_distribution.png")

## 5. Feature Analysis

In [ ]:
# Analyze features
features = df.columns[:-1]  # All except target
print(f"Number of features: {len(features)}")
print("\nFeature names:")
for i, feat in enumerate(features, 1):
    print(f"  {i}. {feat}")

In [ ]:
# Plot feature distributions
n_features = len(features)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3))
axes = axes.ravel()

for idx, feature in enumerate(features):
    df[feature].hist(bins=30, ax=axes[idx], edgecolor='black', alpha=0.7)
    axes[idx].set_title(feature, fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

# Hide empty subplots
for idx in range(len(features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../results/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Plot saved to results/feature_distributions.png")

## 6. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation = df.corr()

print("Correlation Matrix:")
print("="*60)
print(correlation)

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation, dtype=bool))  # Mask upper triangle

sns.heatmap(
    correlation, 
    mask=mask,
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm',
    center=0,
    square=True, 
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"}
)

plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../results/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Plot saved to results/correlation_matrix.png")

In [ ]:
# Analyze correlation with target variable
target_corr = df.corr()['is_phishing'].sort_values(ascending=False)

print("Correlation with Target Variable (is_phishing):")
print("="*60)
print(target_corr)

print("\n\nTop 5 Most Correlated Features:")
print("-"*60)
for i, (feat, corr) in enumerate(target_corr[1:6].items(), 1):
    print(f"{i}. {feat}: {corr:.4f}")